# 01 - EDA e Qualidade de Fontes

Objetivo: medir o calor das fontes com uma base causal e sem inflar sistemas por multiplas aparicoes do mesmo telefone.

Premissas principais:
- Excluir status intermediario `processing`.
- Excluir telefones fixos com filtro case-insensitive.
- Usar entrega como metrica operacional: `delivered` ou `read`.
- Usar leitura como metrica de negocio e diagnostico.
- Deduplicar aparicoes para uma linha por `(telefone_numero, id_sistema)`.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportion_confint

import utils as u

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print('Imports OK')


## 1. Carregamento e filtros

Esta etapa aplica as regras de limpeza antes de qualquer join ou ranking.


In [ ]:
df_disparo_raw, df_telefone_raw = u.carregar_dados()

df_disparo = u.filtrar_status_invalidos(df_disparo_raw)
df_telefone = u.filtrar_telefones_fixos(df_telefone_raw)

print('Disparos raw:', df_disparo_raw.shape)
print('Disparos validos:', df_disparo.shape)
print('Telefones raw:', df_telefone_raw.shape)
print('Telefones validos:', df_telefone.shape)
print('\nStatus validos:')
print(df_disparo['status_disparo'].value_counts(dropna=False))
print('\nTipos de telefone apos filtro:')
print(df_telefone['telefone_tipo'].value_counts(dropna=False))


## 2. Aparicoes: brutas, por fonte e por CPF

A visao por fonte evita que um telefone com muitas aparicoes no mesmo sistema conte varias vezes no ranking da fonte.


In [ ]:
df_aparicoes_brutas = u.explodir_aparicoes(df_telefone)
df_aparicoes_fonte = u.preparar_aparicoes_por_fonte(df_aparicoes_brutas)
df_telefone_cpf = u.preparar_telefone_cpf(df_aparicoes_brutas)

resumo_aparicoes = pd.DataFrame({
    'visao': ['bruta', 'por_fonte', 'telefone_cpf'],
    'linhas': [len(df_aparicoes_brutas), len(df_aparicoes_fonte), len(df_telefone_cpf)],
    'telefones': [df_aparicoes_brutas['telefone_numero'].nunique(), df_aparicoes_fonte['telefone_numero'].nunique(), df_telefone_cpf['telefone_numero'].nunique()],
})
display(resumo_aparicoes)
print('Reducao de linhas bruta -> por_fonte:', f"{1 - len(df_aparicoes_fonte) / len(df_aparicoes_brutas):.2%}")


## 3. Join causal disparo x sistema

Para o ranking, uma fonte so recebe credito se o telefone ja existia naquela fonte antes do envio.


In [ ]:
df_disparo_sistema = u.join_disparo_sistema(df_disparo, df_aparicoes_fonte, causal=True)

colunas_preview = [
    'id_disparo', 'cpf', 'contato_telefone', 'id_sistema',
    'status_disparo', 'envio_datahora', 'registro_data_atualizacao'
]
display(df_disparo_sistema[colunas_preview].head(10))


## 4. Ranking de sistemas por entrega

O ranking usa Wilson lower bound de entrega para penalizar baixo volume.


In [ ]:
metricas_sistema = u.calcular_metricas_sistema(df_disparo_sistema)
metricas_sistema = u.calcular_score_sistema(metricas_sistema)

cols = [
    'id_sistema', 'total_disparos', 'sucessos_entrega', 'taxa_entrega',
    'taxa_leitura', 'taxa_falha', 'wilson_lower_entrega', 'score_sistema'
]
display(metricas_sistema[cols])

top = metricas_sistema.head(15).copy()
top[['ci_low', 'ci_high']] = top.apply(
    lambda row: pd.Series(proportion_confint(row['sucessos_entrega'], row['total_disparos'], alpha=0.05, method='wilson')),
    axis=1,
)

fig, ax = plt.subplots(figsize=(12, 6))
y_pos = np.arange(len(top))
ax.barh(y_pos, top['taxa_entrega'], color='steelblue', alpha=0.85)
ax.errorbar(
    top['taxa_entrega'], y_pos,
    xerr=[top['taxa_entrega'] - top['ci_low'], top['ci_high'] - top['taxa_entrega']],
    fmt='none', color='black', capsize=3, alpha=0.7,
)
ax.set_yticks(y_pos)
ax.set_yticklabels(top['id_sistema'])
ax.invert_yaxis()
ax.set_xlabel('Taxa de entrega')
ax.set_title('Ranking causal de sistemas por entrega (IC Wilson 95%)')
ax.set_xlim(0, 1)
for i, (_, row) in enumerate(top.iterrows()):
    ax.text(min(row['taxa_entrega'] + 0.01, 0.98), i, f"n={row['total_disparos']:,}", va='center', fontsize=8)
plt.tight_layout()
plt.show()


## 5. Decaimento temporal

Avaliamos se a idade do dado reduz a chance de entrega.


In [ ]:
decaimento = u.calcular_decaimento_temporal(df_disparo_sistema)
display(decaimento)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(decaimento))
width = 0.35
ax.bar(x - width/2, decaimento['taxa_entrega'], width, label='Entrega', color='steelblue')
ax.bar(x + width/2, decaimento['taxa_leitura'], width, label='Leitura', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(decaimento['faixa_atualizacao'], rotation=15)
ax.set_ylim(0, 1)
ax.set_ylabel('Taxa')
ax.set_xlabel('Dias desde atualizacao')
ax.set_title('Decaimento da qualidade do dado')
ax.legend()
for i, row in decaimento.iterrows():
    if pd.notna(row['total']):
        ax.text(i, max(row['taxa_entrega'], row['taxa_leitura']) + 0.02, f"n={int(row['total']):,}", ha='center', fontsize=8)
plt.tight_layout()
plt.show()

contingencia = decaimento[['sucessos_entrega']].copy()
contingencia['nao_entrega'] = decaimento['total'] - decaimento['sucessos_entrega']
contingencia = contingencia.dropna()
if len(contingencia) > 1:
    chi2, p_valor, dof, esperado = stats.chi2_contingency(contingencia.values)
    print(f'Chi-square: {chi2:.2f}')
    print(f'p-valor: {p_valor:.2e}')
else:
    print('Sem faixas suficientes para teste chi-square.')


## 6. Sinais de vies de selecao

Comparamos volume, first-touch e vitorias intra-CPF para separar qualidade real de pre-selecao historica.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(metricas_sistema['total_disparos'], metricas_sistema['taxa_entrega'], s=90, alpha=0.7, color='steelblue', edgecolors='white')
log_vol = np.log10(metricas_sistema['total_disparos'])
slope, intercept, r_value, p_value, std_err = stats.linregress(log_vol, metricas_sistema['taxa_entrega'])
x_fit = np.linspace(metricas_sistema['total_disparos'].min(), metricas_sistema['total_disparos'].max(), 100)
y_fit = slope * np.log10(x_fit) + intercept
ax.plot(x_fit, y_fit, 'r--', alpha=0.7, label=f'R2={r_value**2:.3f}, p={p_value:.3f}')
ax.set_xscale('log')
ax.set_xlabel('Total de disparos unicos (log)')
ax.set_ylabel('Taxa de entrega')
ax.set_title('Volume de disparos vs. taxa de entrega')
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.show()

print(f'Correlacao Pearson (log volume vs taxa): R={r_value:.3f}, p={p_value:.3f}')


In [ ]:
df_first = (
    df_disparo.sort_values('envio_datahora')
    .drop_duplicates(subset='contato_telefone', keep='first')
)
df_first_sistema = u.join_disparo_sistema(df_first, df_aparicoes_fonte, causal=True)
metricas_first = u.calcular_score_sistema(u.calcular_metricas_sistema(df_first_sistema))

comparacao_first = metricas_sistema[['id_sistema', 'wilson_lower_entrega', 'taxa_entrega']].merge(
    metricas_first[['id_sistema', 'wilson_lower_entrega', 'taxa_entrega']],
    on='id_sistema', how='outer', suffixes=('_geral', '_first')
)
comparacao_first['rank_geral'] = comparacao_first['wilson_lower_entrega_geral'].rank(ascending=False)
comparacao_first['rank_first'] = comparacao_first['wilson_lower_entrega_first'].rank(ascending=False)
comparacao_first['delta_rank'] = comparacao_first['rank_geral'] - comparacao_first['rank_first']

display(comparacao_first.sort_values('rank_first'))


In [ ]:
cpf_sistema = (
    df_disparo_sistema.dropna(subset=['cpf'])
    .groupby(['cpf', 'id_sistema'])
    .agg(
        total=('id_disparo', 'nunique'),
        entregas=('status_disparo', lambda s: s.isin(u.STATUS_ENTREGA).sum()),
        leituras=('status_disparo', lambda s: (s == 'read').sum()),
    )
    .reset_index()
)
cpf_sistema['taxa_entrega'] = cpf_sistema['entregas'] / cpf_sistema['total']
cpf_multi = cpf_sistema.groupby('cpf')['id_sistema'].nunique().loc[lambda s: s >= 2].index
cpf_rank = cpf_sistema[cpf_sistema['cpf'].isin(cpf_multi)].copy()
cpf_rank['rank_no_cpf'] = cpf_rank.groupby('cpf')['taxa_entrega'].rank(ascending=False, method='first')

vitorias = cpf_rank[cpf_rank['rank_no_cpf'] == 1].groupby('id_sistema').size().reset_index(name='vitorias')
participacoes = cpf_rank.groupby('id_sistema').size().reset_index(name='participacoes')
vitorias = vitorias.merge(participacoes, on='id_sistema', how='outer').fillna(0)
vitorias['taxa_vitoria'] = np.where(vitorias['participacoes'] > 0, vitorias['vitorias'] / vitorias['participacoes'], np.nan)
vitorias = vitorias.sort_values('taxa_vitoria', ascending=False)

display(vitorias)


## 7. Artefatos da Parte 1


In [ ]:
metricas_final = metricas_sistema.merge(
    metricas_first[['id_sistema', 'wilson_lower_entrega']].rename(columns={'wilson_lower_entrega': 'wilson_first_touch_entrega'}),
    on='id_sistema', how='left'
).merge(
    vitorias[['id_sistema', 'taxa_vitoria']],
    on='id_sistema', how='left'
)

u.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
artefatos = {
    'metricas_sistema': metricas_final,
    'decaimento': decaimento,
    'metricas_first_touch': metricas_first,
    'vitorias_intra_cpf': vitorias,
}
for nome, obj in artefatos.items():
    caminho = u.PROCESSED_DIR / f'{nome}.pkl'
    with open(caminho, 'wb') as f:
        pickle.dump(obj, f)
    print(f'{nome} salvo em {caminho}')

display(metricas_final)
